# 01 – Dataset EDA (Stage 1.1)

Exploratory analysis of the Kaggle dataset `paultimothymooney/chest-xray-pneumonia`.

**How to run on Kaggle:** attach the dataset *Chest X-Ray Images (Pneumonia)* (Add Input → search `chest-xray-pneumonia`), turn **Internet on** (needed to clone the repo), then Run All. No GPU needed.

Outputs are saved to `results/stage1_eda/`. Decisions behind the patient-ID rule and the folders used: issue #21.

In [ ]:
# Setup: on Kaggle, clone the repo; locally, run from the repo root.
import os, sys, subprocess

REPO_URL = "https://github.com/Abhishek-Singh-2008/Pneumonia-detection.git"
BRANCH = "stage1-data-baseline"  # switch to "main" once Stage 1 is merged

if os.path.isdir("/kaggle/working"):
    repo_dir = "/kaggle/working/Pneumonia-detection"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, repo_dir], check=True)
    os.chdir(repo_dir)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

## 1. Environment
Versions are printed so `requirements.txt` matches what the code was run with.

In [ ]:
import platform
import numpy as np, pandas as pd, matplotlib, sklearn, PIL
import matplotlib.pyplot as plt
from PIL import Image

versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__, "Pillow": PIL.__version__}
try:
    import torch, torchvision
    versions.update({"torch": torch.__version__, "torchvision": torchvision.__version__})
except ImportError:
    pass
for k, v in versions.items():
    print(f"{k}=={v}")

In [ ]:
from src.config import SEED, get_data_root, RESULTS_DIR
from src.data.index import build_index
from src.utils import set_seed

set_seed(SEED)
DATA_ROOT = get_data_root()
OUT = RESULTS_DIR / "stage1_eda"
OUT.mkdir(parents=True, exist_ok=True)
print("Data root:", DATA_ROOT)
print("Contents:", sorted(p.name for p in DATA_ROOT.iterdir()))

## 2. Image index
One row per image. Only the top-level `chest_xray/{train,val,test}` folders are read; the duplicate `chest_xray/chest_xray/` copy and `__MACOSX/` are ignored.

In [ ]:
index = build_index(DATA_ROOT)
print(len(index), "images")
index.head()

## 3. Class counts per official folder

In [ ]:
counts = (index.pivot_table(index="official_folder", columns="label_name", values="image_id", aggfunc="count")
               .reindex(["train", "val", "test"]))
counts["total"] = counts.sum(axis=1)
counts["% pneumonia"] = (100 * counts["PNEUMONIA"] / counts["total"]).round(1)
counts.loc["all"] = counts[["NORMAL", "PNEUMONIA", "total"]].sum().tolist() + [round(100 * counts["PNEUMONIA"].sum() / counts["total"].sum(), 1)]
counts.to_csv(OUT / "class_counts.csv")

# Sanity check against the official numbers (issue #21)
assert counts.loc["train", ["NORMAL", "PNEUMONIA"]].tolist() == [1341, 3875]
assert counts.loc["val", ["NORMAL", "PNEUMONIA"]].tolist() == [8, 8]
assert counts.loc["test", ["NORMAL", "PNEUMONIA"]].tolist() == [234, 390]
counts

In [ ]:
ax = counts.loc[["train", "val", "test"], ["NORMAL", "PNEUMONIA"]].plot.bar(figsize=(6, 4), rot=0)
ax.set_ylabel("Images"); ax.set_title("Images per class and official folder")
for c in ax.containers:
    ax.bar_label(c)
plt.tight_layout(); plt.savefig(OUT / "class_counts.png", dpi=150); plt.show()

## 4. Patients
Patient ID rule (issue #21): `PNEUMONIA_person{N}_{bacteria|virus}`, `NORMAL_IM-{XXXX}`, `NORMAL_NORMAL2-IM-{XXXX}`.
Several images can belong to one patient, so the validation split must be made at patient level (Stage 1.2).

In [ ]:
patients = index.groupby(["official_folder", "label_name"]).agg(images=("image_id", "count"), patients=("patient_id", "nunique"))
patients["images per patient"] = (patients["images"] / patients["patients"]).round(2)
patients.to_csv(OUT / "patients.csv")

# The official folders share no patients under this rule
folders = {f: set(index.loc[index.official_folder == f, "patient_id"]) for f in ["train", "val", "test"]}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    print(f"patients in both {a} and {b}: {len(folders[a] & folders[b])}")
patients

In [ ]:
per_patient = index.groupby(["label_name", "patient_id"]).size()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, cls in zip(axes, ["NORMAL", "PNEUMONIA"]):
    vc = per_patient[cls].value_counts().sort_index()
    ax.bar(vc.index.astype(str), vc.values)
    ax.set_title(f"{cls}: images per patient"); ax.set_xlabel("images"); ax.set_ylabel("patients")
plt.tight_layout(); plt.savefig(OUT / "images_per_patient.png", dpi=150); plt.show()
print("Max images for one patient:", per_patient.max(), per_patient.idxmax())

## 5. Image sizes and colour modes
Read from the file headers only (fast). Sizes vary a lot, which is why every model uses the same plain resize to 224×224 (issue #21, row 5).
Some files are not single-channel grayscale, so all images are converted to 3 channels.

In [ ]:
sizes = []
for rel in index["filepath"]:
    with Image.open(DATA_ROOT / rel) as im:
        sizes.append((im.width, im.height, im.mode))
index[["width", "height", "mode"]] = pd.DataFrame(sizes, index=index.index)
index["aspect"] = (index["width"] / index["height"]).round(3)

size_summary = index.groupby("label_name")[["width", "height", "aspect"]].describe().T.round(1)
size_summary.to_csv(OUT / "image_size_summary.csv")
modes = index.groupby(["label_name", "mode"]).size().unstack(fill_value=0)
modes.to_csv(OUT / "image_modes.csv")
display(modes)
size_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for cls, colour in [("NORMAL", "tab:blue"), ("PNEUMONIA", "tab:orange")]:
    sub = index[index.label_name == cls]
    axes[0].scatter(sub.width, sub.height, s=3, alpha=0.4, label=cls, color=colour)
    axes[1].hist(sub.aspect, bins=50, alpha=0.6, label=cls, color=colour)
axes[0].set_xlabel("width (px)"); axes[0].set_ylabel("height (px)"); axes[0].set_title("Image sizes"); axes[0].legend()
axes[1].set_xlabel("width / height"); axes[1].set_ylabel("images"); axes[1].set_title("Aspect ratio"); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT / "image_sizes.png", dpi=150); plt.show()

## 6. Sample images
Random training images (fixed seed). Look out for text markers, L/R letters and borders: these are the possible shortcuts this project measures.

In [ ]:
train = index[index.official_folder == "train"]
groups = {
    "NORMAL": train[train.label_name == "NORMAL"],
    "PNEUMONIA (bacteria)": train[train.patient_id.str.endswith("_bacteria")],
    "PNEUMONIA (virus)": train[train.patient_id.str.endswith("_virus")],
}
n = 5
fig, axes = plt.subplots(len(groups), n, figsize=(3 * n, 3.2 * len(groups)))
for row, (name, df) in enumerate(groups.items()):
    for col, (_, r) in enumerate(df.sample(n, random_state=SEED).iterrows()):
        with Image.open(DATA_ROOT / r.filepath) as im:
            axes[row, col].imshow(im.convert("L"), cmap="gray")
        axes[row, col].set_title(f"{name}\n{r.width}×{r.height}", fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout(); plt.savefig(OUT / "sample_grid.png", dpi=120); plt.show()

## 7. Summary
- 5,856 images: train 1,341 normal / 3,875 pneumonia, official val 8 / 8, test 234 / 390.
- The training data is about 74% pneumonia, so class imbalance is handled with `pos_weight` (Stage 1.2).
- The official val folder has only 16 images, so a patient-level validation split is made from train + val (Stage 1.2).
- Image sizes and aspect ratios vary widely, so all images are resized to 224×224 and converted to 3 channels (Stage 1.3).
- The official folders share no patients under the patient-ID rule.